# Repeated cross-validation, final models, and temporal projection

This notebook reproduces the complete TF-IDF and BERTimbau experiments from the
released gold files. It writes new outputs to `reproduced_results/` and does not
overwrite the paper-facing files under `results/`.

Full transformer execution requires the dependencies in
`requirements-training.txt`, network access or a cached BERTimbau model, and a
CUDA-capable GPU for practical runtimes.


In [ ]:
# ============================================================
# 0. Configuration
# ============================================================

from pathlib import Path
import json
import gc
import warnings
import re
import math
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "gold").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run the notebook from inside the repository."
    )

REPO_ROOT = find_repo_root()

# Reproduction outputs are written separately from the released results.
RESULTS_DIR = Path(
    os.environ.get(
        "AXIOLOGICAL_RESULTS_DIR",
        str(REPO_ROOT / "reproduced_results"),
    )
)
TABLE_DIR = RESULTS_DIR / "tables"
FIG_DIR = RESULTS_DIR / "figures"
PRED_DIR = RESULTS_DIR / "predictions"
REPORT_DIR = RESULTS_DIR / "reports"
RUNTIME_DIR = RESULTS_DIR / "runtime_no_checkpoints"

for directory in [
    RESULTS_DIR,
    TABLE_DIR,
    FIG_DIR,
    PRED_DIR,
    REPORT_DIR,
    RUNTIME_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Reproduction output:", RESULTS_DIR.resolve())

APP_FILE_NAME = "applicability_gold_final_1497.csv"
AXIS3_FILE_NAME = "axis_gold_consensus_3class_only.csv"
AXIS5_FILE_NAME = "axis_gold_consensus_5class_only.csv"
PAIRWISE_FILE_NAME = "final_pairwise_annotations_1497.csv"
IAA_FILE_NAME = "iaa_metrics_final_1497.csv"
EXTERNAL_2024_FILE_NAME = "corpus_2024.csv"

INPUT_SEARCH_DIRS = [
    REPO_ROOT / "data" / "gold",
    REPO_ROOT / "data" / "external_2024",
    REPO_ROOT,
]

RUN_TFIDF_SHARED_SPLITS = True
RUN_BERTIMBAU_REPEATED_CV = True
RUN_FINE_5CLASS_TASKS = True
RUN_BERTIMBAU_ON_FINE_5CLASS = False
RUN_EXTERNAL_2024_PROJECTION = True
RUN_LINEAR_ONLY_PROJECTION = True
RUN_HYBRID_BEST_PROJECTION = True
RUN_FINAL_BERTIMBAU_FOR_HYBRID_PROJECTION = True
HYBRID_POLICY = "previous_decision"

N_SPLITS = 5
SPLIT_SEEDS = [42, 2026, 7]

BERTIMBAU_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 160
EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
RANDOM_STATE = 42

APPLICABILITY_ORDER = [0, 1]
ACCEPTANCE_ORDER_3 = [-1, 0, 1]
INTENSITY_ORDER_3 = [0, 1, 2]
ACCEPTANCE_ORDER_5 = [-2, -1, 0, 1, 2]
INTENSITY_ORDER_5 = [1, 2, 3, 4, 5]

APP_TO_INDEX = {0: 0, 1: 1}
INDEX_TO_APP = {v: k for k, v in APP_TO_INDEX.items()}
ACC3_TO_INDEX = {-1: 0, 0: 1, 1: 2}
INDEX_TO_ACC3 = {v: k for k, v in ACC3_TO_INDEX.items()}
INT3_TO_INDEX = {0: 0, 1: 1, 2: 2}
INDEX_TO_INT3 = {v: k for k, v in INT3_TO_INDEX.items()}
ACC5_TO_INDEX = {-2: 0, -1: 1, 0: 2, 1: 3, 2: 4}
INDEX_TO_ACC5 = {v: k for k, v in ACC5_TO_INDEX.items()}
INT5_TO_INDEX = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
INDEX_TO_INT5 = {v: k for k, v in INT5_TO_INDEX.items()}


In [ ]:
# ============================================================
# 1. Utility functions
# ============================================================

def locate_file(filename, search_dirs=INPUT_SEARCH_DIRS):
    """
    Locate a required file. Prefer files in the project root, then known folders,
    then a shallow recursive search.
    """
    for d in search_dirs:
        p = d / filename
        if p.exists():
            return p

    matches = list(Path(".").rglob(filename))
    matches = [p for p in matches if ".ipynb_checkpoints" not in str(p)]
    if matches:
        matches = sorted(matches, key=lambda p: (len(p.parts), str(p)))
        return matches[0]

    raise FileNotFoundError(f"Could not locate required file: {filename}")

def maybe_locate_file(filename, search_dirs=INPUT_SEARCH_DIRS):
    try:
        return locate_file(filename, search_dirs)
    except FileNotFoundError:
        return None

def norm_cell(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    return "" if s.lower() in {"nan", "none", "null"} else s

def to_int_series(s):
    return pd.to_numeric(s, errors="coerce").astype("Int64")

def clean_for_feature_analysis(text):
    s = norm_cell(text).lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"[@#]\w+", " ", s)
    s = re.sub(r"[^a-záàâãéèêíïóôõöúçñ0-9_\s-]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

PT_STOPWORDS = {
    "a","à","ao","aos","as","às","o","os","um","uma","uns","umas",
    "de","da","das","do","dos","em","no","na","nos","nas","por","para","pra","com","sem",
    "sobre","entre","até","apos","após","antes","desde","contra","como","que","quem","qual","quais",
    "quando","onde","porque","porquê","pq","se","ser","sendo","foi","foram","era","eram","é","são",
    "está","estao","estão","estar","tem","têm","ter","vai","vão","vou","vamos","pode","podem",
    "mais","menos","muito","muita","muitos","muitas","pouco","pouca","também","tambem","já","ja",
    "não","nao","sim","só","so","isso","isto","essa","esse","esses","essas","este","esta","estas",
    "ele","ela","eles","elas","eu","tu","você","voce","vocês","voces","meu","minha","meus","minhas",
    "seu","sua","seus","suas","nosso","nossa","nossos","nossas","lhe","lhes","me","te","nos","se",
    "e","ou","mas","pois","então","entao","logo","assim","ainda","mesmo","mesma","mesmos","mesmas",
    "todo","toda","todos","todas","cada","algum","alguma","alguns","algumas","outro","outra","outros","outras",
    "www","http","https","br","com","html","amp","link","via",
}

DOMAIN_KEEP = {
    "ia", "ai", "chatgpt", "gpt", "llm", "llms", "gemini", "claude", "copilot",
    "inteligência", "inteligencia", "artificial", "educação", "educacao", "ensino",
    "professor", "professores", "aluno", "alunos", "estudante", "estudantes",
    "escola", "universidade", "faculdade", "aprendizagem", "aula", "aulas",
}

def remove_pt_stopwords_text(text):
    toks = clean_for_feature_analysis(text).split()
    kept = []
    for t in toks:
        if t in DOMAIN_KEEP:
            kept.append(t)
        elif t not in PT_STOPWORDS and len(t) > 2:
            kept.append(t)
    return " ".join(kept)

def compute_metrics(y_true, y_pred, labels, ordinal=True):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
    }
    if ordinal:
        out["mae_ordinal"] = mean_absolute_error(y_true, y_pred)
        out["linear_weighted_kappa"] = cohen_kappa_score(y_true, y_pred, labels=labels, weights="linear")
        out["quadratic_weighted_kappa"] = cohen_kappa_score(y_true, y_pred, labels=labels, weights="quadratic")
    else:
        out["mae_ordinal"] = np.nan
        out["linear_weighted_kappa"] = np.nan
        out["quadratic_weighted_kappa"] = np.nan
    return out

def task_labels_and_ordinal(task):
    if task == "applicability":
        return APPLICABILITY_ORDER, False
    if task == "acceptance_3":
        return ACCEPTANCE_ORDER_3, True
    if task == "intensity_3":
        return INTENSITY_ORDER_3, True
    if task == "acceptance_5":
        return ACCEPTANCE_ORDER_5, True
    if task == "intensity_5":
        return INTENSITY_ORDER_5, True
    raise ValueError(task)

def task_mapping(task):
    if task == "applicability":
        return APP_TO_INDEX, INDEX_TO_APP
    if task == "acceptance_3":
        return ACC3_TO_INDEX, INDEX_TO_ACC3
    if task == "intensity_3":
        return INT3_TO_INDEX, INDEX_TO_INT3
    if task == "acceptance_5":
        return ACC5_TO_INDEX, INDEX_TO_ACC5
    if task == "intensity_5":
        return INT5_TO_INDEX, INDEX_TO_INT5
    raise ValueError(task)

def make_splits(y, seed, n_splits=N_SPLITS):
    min_count = int(pd.Series(y).value_counts().min())
    effective_splits = min(n_splits, min_count)
    if effective_splits < 2:
        raise ValueError(f"Not enough examples for stratified CV. min_count={min_count}")
    skf = StratifiedKFold(n_splits=effective_splits, shuffle=True, random_state=seed)
    return list(skf.split(np.zeros(len(y)), y))

def load_json_or_jsonl(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in [".jsonl", ".ndjson"]:
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))
        return pd.DataFrame(rows)

    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, list):
        return pd.DataFrame(data)
    if isinstance(data, dict):
        for key in ["data", "posts", "items", "records"]:
            if key in data and isinstance(data[key], list):
                return pd.DataFrame(data[key])
        return pd.DataFrame([data])
    raise ValueError(f"Unsupported JSON structure in {path}")

In [ ]:
# ============================================================
# 2. Load final consolidated data and audit
# ============================================================

app_path = locate_file(APP_FILE_NAME)
axis3_path = locate_file(AXIS3_FILE_NAME)
axis5_path = locate_file(AXIS5_FILE_NAME)
pairwise_path = locate_file(PAIRWISE_FILE_NAME)
iaa_path = maybe_locate_file(IAA_FILE_NAME)

print("Located final files:")
print("applicability:", app_path)
print("axis 3:", axis3_path)
print("axis 5:", axis5_path)
print("pairwise:", pairwise_path)
print("IAA:", iaa_path)

app_df = pd.read_csv(app_path)
axis3_df = pd.read_csv(axis3_path)
axis5_df = pd.read_csv(axis5_path)
pairwise_df = pd.read_csv(pairwise_path)
iaa_df = pd.read_csv(iaa_path) if iaa_path is not None else pd.DataFrame()

# Standardize and cast.
for df in [app_df, axis3_df, axis5_df, pairwise_df]:
    df.columns = [str(c).strip() for c in df.columns]
    if "text" in df.columns:
        df["text"] = df["text"].fillna("").astype(str)
    if "item_id" in df.columns:
        df["item_id"] = to_int_series(df["item_id"]).astype(int)

app_df["applicability_score"] = to_int_series(app_df["applicability_score"]).astype(int)

axis3_df["applicability_score"] = to_int_series(axis3_df["applicability_score"]).astype(int)
axis3_df["acceptance_3"] = to_int_series(axis3_df["acceptance_3"]).astype(int)
axis3_df["intensity_3"] = to_int_series(axis3_df["intensity_3"]).astype(int)

axis5_df["applicability_score"] = to_int_series(axis5_df["applicability_score"]).astype(int)
axis5_df["acceptance_5"] = to_int_series(axis5_df["acceptance_5"]).astype(int)
axis5_df["intensity_5"] = to_int_series(axis5_df["intensity_5"]).astype(int)

# Clean variants.
for df in [app_df, axis3_df, axis5_df]:
    df["text_clean"] = df["text"].apply(clean_for_feature_analysis)
    df["text_no_stop"] = df["text"].apply(remove_pt_stopwords_text)

audit_rows = [
    {"dataset": "applicability_gold_final_1497", "n": len(app_df), "unique_item_ids": app_df["item_id"].nunique()},
    {"dataset": "axis_gold_consensus_3class_only", "n": len(axis3_df), "unique_item_ids": axis3_df["item_id"].nunique()},
    {"dataset": "axis_gold_consensus_5class_only", "n": len(axis5_df), "unique_item_ids": axis5_df["item_id"].nunique()},
    {"dataset": "final_pairwise_annotations_1497", "n": len(pairwise_df), "unique_item_ids": pairwise_df["item_id"].nunique()},
    {"dataset": "final_applicable_count", "n": int((app_df["applicability_score"] == 1).sum()), "unique_item_ids": np.nan},
    {"dataset": "final_excluded_count", "n": int((app_df["applicability_score"] == 0).sum()), "unique_item_ids": np.nan},
]

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(TABLE_DIR / "final_input_audit.csv", index=False)

# Copy final IAA metrics into results.
if len(iaa_df) > 0:
    iaa_df.to_csv(TABLE_DIR / "iaa_metrics_final_1497.csv", index=False)

display(audit_df)
display(iaa_df if len(iaa_df) > 0 else pd.DataFrame({"note": ["IAA file not found"]}))

# Hard checks for final stage.
assert len(app_df) == 1497, f"Expected 1497 applicability rows, got {len(app_df)}"
assert app_df["item_id"].nunique() == 1497, "Applicability item_id count is not 1497."
assert set(app_df["applicability_score"].unique()).issubset({0, 1}), "Invalid applicability labels."
assert (axis3_df["applicability_score"] == 1).all(), "axis3 dataset must only contain applicable items."
assert (axis5_df["applicability_score"] == 1).all(), "axis5 dataset must only contain applicable items."

print("Final consolidated input audit passed.")

In [ ]:
# ============================================================
# 3. Final matrices and figures
# ============================================================

def crosstab_matrix(df, y_col, x_col, y_order, x_order):
    mat = pd.crosstab(df[y_col], df[x_col])
    return mat.reindex(index=y_order, columns=x_order, fill_value=0)

def plot_bubble_matrix(counts, title, output_path, kind="5x5"):
    fig, ax = plt.subplots(figsize=(8.8, 6.6))

    xs, ys, ns = [], [], []
    for y_val in counts.index:
        for x_val in counts.columns:
            n = int(counts.loc[y_val, x_val])
            if n > 0:
                xs.append(int(x_val))
                ys.append(int(y_val))
                ns.append(n)

    max_n = max(ns) if ns else 1
    sizes = [max(70, 2000 * (n / max_n)) for n in ns]
    ax.scatter(xs, ys, s=sizes, alpha=0.65, edgecolors="black", linewidths=0.8)

    for x, y, n in zip(xs, ys, ns):
        ax.text(x, y, str(n), ha="center", va="center", fontsize=9)

    ax.grid(True, linestyle=":", alpha=0.35)
    ax.set_title(title)

    if kind == "3x3":
        ax.set_xlim(-1.6, 1.6)
        ax.set_ylim(-0.5, 2.5)
        ax.set_xticks(ACCEPTANCE_ORDER_3)
        ax.set_yticks(INTENSITY_ORDER_3)
        ax.set_xticklabels(["negative", "neutral", "positive"])
        ax.set_yticklabels(["low", "medium", "high"])
        ax.set_xlabel("Acceptance, 3 classes")
        ax.set_ylabel("Intensity, 3 classes")
        ax.axvline(0, linestyle=":", linewidth=1.2)
    else:
        ax.set_xlim(-2.6, 2.6)
        ax.set_ylim(0.5, 5.5)
        ax.set_xticks(ACCEPTANCE_ORDER_5)
        ax.set_yticks(INTENSITY_ORDER_5)
        ax.set_xticklabels([
            "-2\nstrong\nrejection",
            "-1\ncritical\nrejection",
            "0\nneutral",
            "1\nprudent\nacceptance",
            "2\nstrong\nacceptance",
        ])
        ax.set_xlabel("Acceptance, 5 classes")
        ax.set_ylabel("Intensity, 5 classes")
        ax.axvline(0, linestyle=":", linewidth=1.2)

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

# Consensus gold matrices.
consensus_3_mat = crosstab_matrix(axis3_df, "intensity_3", "acceptance_3", INTENSITY_ORDER_3, ACCEPTANCE_ORDER_3)
consensus_5_mat = crosstab_matrix(axis5_df, "intensity_5", "acceptance_5", INTENSITY_ORDER_5, ACCEPTANCE_ORDER_5)

consensus_3_mat.to_csv(TABLE_DIR / "consensus_gold_3x3_matrix.csv")
consensus_5_mat.to_csv(TABLE_DIR / "consensus_gold_5x5_matrix.csv")

plot_bubble_matrix(consensus_3_mat, "Consensus gold 3x3 matrix", FIG_DIR / "consensus_gold_3x3_matrix.png", kind="3x3")
plot_bubble_matrix(consensus_5_mat, "Consensus gold 5x5 matrix", FIG_DIR / "consensus_gold_5x5_matrix.png", kind="5x5")

# A1/A2 matrices from pairwise annotations, for descriptive reporting.
app_pairwise = pairwise_df[pairwise_df.get("final_applicability_score", pairwise_df.get("both_applicable", 0)).fillna(0).astype(float).astype(int) == 1].copy()

for prefix in ["a1", "a2"]:
    for col in [f"{prefix}_acceptance_5", f"{prefix}_intensity_5", f"{prefix}_acceptance_3", f"{prefix}_intensity_3"]:
        if col in app_pairwise.columns:
            app_pairwise[col] = pd.to_numeric(app_pairwise[col], errors="coerce")

    df5 = app_pairwise.dropna(subset=[f"{prefix}_acceptance_5", f"{prefix}_intensity_5"]).copy()
    df3 = app_pairwise.dropna(subset=[f"{prefix}_acceptance_3", f"{prefix}_intensity_3"]).copy()

    if len(df5) > 0:
        df5[f"{prefix}_acceptance_5"] = df5[f"{prefix}_acceptance_5"].astype(int)
        df5[f"{prefix}_intensity_5"] = df5[f"{prefix}_intensity_5"].astype(int)
        mat5 = crosstab_matrix(df5, f"{prefix}_intensity_5", f"{prefix}_acceptance_5", INTENSITY_ORDER_5, ACCEPTANCE_ORDER_5)
        mat5.to_csv(TABLE_DIR / f"{prefix}_final_5x5_matrix.csv")
        plot_bubble_matrix(mat5, f"{prefix.upper()} final 5x5 matrix", FIG_DIR / f"{prefix}_final_5x5_matrix.png", kind="5x5")

    if len(df3) > 0:
        df3[f"{prefix}_acceptance_3"] = df3[f"{prefix}_acceptance_3"].astype(int)
        df3[f"{prefix}_intensity_3"] = df3[f"{prefix}_intensity_3"].astype(int)
        mat3 = crosstab_matrix(df3, f"{prefix}_intensity_3", f"{prefix}_acceptance_3", INTENSITY_ORDER_3, ACCEPTANCE_ORDER_3)
        mat3.to_csv(TABLE_DIR / f"{prefix}_final_3x3_matrix.csv")
        plot_bubble_matrix(mat3, f"{prefix.upper()} final 3x3 matrix", FIG_DIR / f"{prefix}_final_3x3_matrix.png", kind="3x3")

display(consensus_3_mat)
display(consensus_5_mat)

In [ ]:
# ============================================================
# 4. Build task datasets
# ============================================================

TASKS_TO_RUN = ["applicability", "acceptance_3", "intensity_3"]

if RUN_FINE_5CLASS_TASKS:
    TASKS_TO_RUN += ["acceptance_5", "intensity_5"]

BERT_TASKS = ["applicability", "acceptance_3", "intensity_3"]
if RUN_BERTIMBAU_ON_FINE_5CLASS:
    BERT_TASKS += ["acceptance_5", "intensity_5"]

def get_task_dataset(task):
    if task == "applicability":
        data = app_df.copy()
        y_col = "applicability_score"
    elif task == "acceptance_3":
        data = axis3_df.copy()
        y_col = "acceptance_3"
    elif task == "intensity_3":
        data = axis3_df.copy()
        y_col = "intensity_3"
    elif task == "acceptance_5":
        data = axis5_df.copy()
        y_col = "acceptance_5"
    elif task == "intensity_5":
        data = axis5_df.copy()
        y_col = "intensity_5"
    else:
        raise ValueError(task)

    data = data.reset_index(drop=True)
    X_raw = data["text"].fillna("").astype(str)
    X_clean = data["text_no_stop"].fillna("").astype(str)
    y = data[y_col].astype(int)
    return data, X_raw, X_clean, y

task_distribution_rows = []

for task in TASKS_TO_RUN:
    data, _, _, y = get_task_dataset(task)
    counts = y.value_counts().sort_index().to_dict()
    print(task, "n:", len(data), "classes:", counts)
    for label, n in counts.items():
        task_distribution_rows.append({"task": task, "label": label, "n": n})

task_distribution_df = pd.DataFrame(task_distribution_rows)
task_distribution_df.to_csv(TABLE_DIR / "task_class_distributions.csv", index=False)
display(task_distribution_df)

In [ ]:
# ============================================================
# 5. TF-IDF shared-split evaluation
# ============================================================

def make_tfidf_model(model_name):
    if "char_logreg" in model_name:
        return Pipeline([
            ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=50000)),
            ("clf", LogisticRegression(max_iter=3000, class_weight="balanced")),
        ])
    if "word_logreg" in model_name:
        return Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000)),
            ("clf", LogisticRegression(max_iter=3000, class_weight="balanced")),
        ])
    if "word_linearsvc" in model_name:
        return Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000)),
            ("clf", LinearSVC(class_weight="balanced")),
        ])
    raise ValueError(model_name)

def tfidf_model_names(input_variant):
    return [
        f"tfidf_word_logreg_{input_variant}",
        f"tfidf_char_logreg_{input_variant}",
        f"tfidf_word_linearsvc_{input_variant}",
    ]

tfidf_rows = []
tfidf_pred_rows = []

if RUN_TFIDF_SHARED_SPLITS:
    for task in TASKS_TO_RUN:
        data, X_raw, X_clean, y = get_task_dataset(task)
        labels, ordinal = task_labels_and_ordinal(task)

        for seed in SPLIT_SEEDS:
            splits = make_splits(y, seed, N_SPLITS)

            for input_variant, X in [("raw", X_raw), ("stopfiltered", X_clean)]:
                for model_name in tfidf_model_names(input_variant):
                    all_true, all_pred = [], []

                    for fold_id, (tr, te) in enumerate(splits):
                        pipe = make_tfidf_model(model_name)
                        pipe.fit(X.iloc[tr], y.iloc[tr])
                        pred = pipe.predict(X.iloc[te])
                        y_true = y.iloc[te].to_numpy()

                        all_true.extend(list(y_true))
                        all_pred.extend(list(pred))

                        for local_i, idx in enumerate(te):
                            tfidf_pred_rows.append({
                                "task": task,
                                "seed": seed,
                                "fold": fold_id,
                                "model": model_name,
                                "input": input_variant,
                                "item_id": int(data.iloc[idx]["item_id"]),
                                "gold": int(y_true[local_i]),
                                "pred": int(pred[local_i]),
                            })

                    metrics = compute_metrics(np.array(all_true), np.array(all_pred), labels, ordinal=ordinal)
                    metrics.update({
                        "task": task,
                        "seed": seed,
                        "model": model_name,
                        "input": input_variant,
                        "family": "tfidf",
                    })
                    tfidf_rows.append(metrics)

tfidf_results_df = pd.DataFrame(tfidf_rows)
tfidf_predictions_df = pd.DataFrame(tfidf_pred_rows)

tfidf_results_df.to_csv(TABLE_DIR / "tfidf_shared_splits_results_by_seed.csv", index=False)
tfidf_predictions_df.to_csv(PRED_DIR / "tfidf_shared_splits_predictions.csv", index=False)

if len(tfidf_results_df) > 0:
    tfidf_summary = (
        tfidf_results_df
        .groupby(["task", "model", "input", "family"], dropna=False)
        .agg(
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_std=("macro_f1", "std"),
            weighted_f1_mean=("weighted_f1", "mean"),
            weighted_f1_std=("weighted_f1", "std"),
            accuracy_mean=("accuracy", "mean"),
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            qwk_mean=("quadratic_weighted_kappa", "mean"),
            qwk_std=("quadratic_weighted_kappa", "std"),
            mae_mean=("mae_ordinal", "mean"),
            mae_std=("mae_ordinal", "std"),
        )
        .reset_index()
    )
else:
    tfidf_summary = pd.DataFrame()

tfidf_summary.to_csv(TABLE_DIR / "tfidf_shared_splits_summary.csv", index=False)
display(tfidf_summary.sort_values(["task", "macro_f1_mean"], ascending=[True, False]))

In [ ]:
# ============================================================
# 6. BERTimbau weighted repeated CV
# ============================================================

def transformers_available():
    try:
        import torch
        import transformers
        import datasets
        return True
    except Exception as e:
        print("Transformers stack unavailable:", repr(e))
        print("Install if needed: pip install transformers datasets accelerate torch")
        return False

def class_weights_from_labels(labels_idx, num_labels):
    counts = np.bincount(labels_idx, minlength=num_labels)
    total = counts.sum()
    weights = total / (num_labels * np.maximum(counts, 1))
    return weights.astype(np.float32), counts

def run_bertimbau_one_fold(task, seed, fold_id, train_texts, train_y_orig, dev_texts, dev_y_orig):
    import torch
    from torch import nn
    from datasets import Dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer,
        DataCollatorWithPadding,
        set_seed,
    )

    set_seed(seed)

    labels_order, ordinal = task_labels_and_ordinal(task)
    label_to_index, index_to_label = task_mapping(task)

    train_y_idx = np.array([label_to_index[int(x)] for x in train_y_orig])
    dev_y_idx = np.array([label_to_index[int(x)] for x in dev_y_orig])
    num_labels = len(label_to_index)

    weights_np, counts = class_weights_from_labels(train_y_idx, num_labels)
    weights = torch.tensor(weights_np, dtype=torch.float)

    tokenizer = AutoTokenizer.from_pretrained(BERTIMBAU_MODEL_NAME)

    ds_train = Dataset.from_dict({"text": list(train_texts), "label": train_y_idx})
    ds_dev = Dataset.from_dict({"text": list(dev_texts), "label": dev_y_idx})

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

    ds_train = ds_train.map(tok, batched=True)
    ds_dev = ds_dev.map(tok, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(BERTIMBAU_MODEL_NAME, num_labels=num_labels)

    out_dir = RUNTIME_DIR / f"bert_cv_{task}_seed{seed}_fold{fold_id}"
    out_dir.mkdir(parents=True, exist_ok=True)

    kwargs = dict(
        output_dir=str(out_dir),
        save_strategy="no",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        report_to=[],
        fp16=torch.cuda.is_available(),
        seed=seed,
    )

    try:
        args = TrainingArguments(eval_strategy="no", **kwargs)
    except TypeError:
        args = TrainingArguments(evaluation_strategy="no", **kwargs)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.get("labels")
            outputs = model(**inputs)
            logits = outputs.get("logits")
            loss_fct = nn.CrossEntropyLoss(weight=weights.to(logits.device))
            loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
    )

    trainer.train()
    pred_out = trainer.predict(ds_dev)
    pred_idx = np.argmax(pred_out.predictions, axis=-1)

    y_true = np.array([index_to_label[int(i)] for i in dev_y_idx])
    y_pred = np.array([index_to_label[int(i)] for i in pred_idx])

    metrics = compute_metrics(y_true, y_pred, labels_order, ordinal=ordinal)
    metrics.update({
        "task": task,
        "seed": seed,
        "fold": fold_id,
        "model": "bertimbau_weighted",
        "family": "transformer",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "train_class_counts": json.dumps(counts.tolist()),
        "train_class_weights": json.dumps(weights_np.tolist()),
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    })

    pred_rows = [{"task": task, "seed": seed, "fold": fold_id, "model": "bertimbau_weighted", "gold": int(t), "pred": int(p)} for t, p in zip(y_true, y_pred)]

    del trainer, model, tokenizer, ds_train, ds_dev
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics, pred_rows

bert_rows = []
bert_pred_rows = []

if RUN_BERTIMBAU_REPEATED_CV and transformers_available():
    for task in BERT_TASKS:
        data, X_raw, _, y = get_task_dataset(task)
        print("Running BERTimbau:", task, "n=", len(data), "classes=", y.value_counts().sort_index().to_dict())

        for seed in SPLIT_SEEDS:
            splits = make_splits(y, seed, N_SPLITS)
            for fold_id, (tr, te) in enumerate(splits):
                print(f"Task={task}, seed={seed}, fold={fold_id}")
                metrics, preds = run_bertimbau_one_fold(
                    task,
                    seed,
                    fold_id,
                    X_raw.iloc[tr].tolist(),
                    y.iloc[tr].tolist(),
                    X_raw.iloc[te].tolist(),
                    y.iloc[te].tolist(),
                )
                bert_rows.append(metrics)

                for local_i, row in enumerate(preds):
                    idx = te[local_i]
                    row["item_id"] = int(data.iloc[idx]["item_id"])
                    bert_pred_rows.append(row)

bert_results_df = pd.DataFrame(bert_rows)
bert_predictions_df = pd.DataFrame(bert_pred_rows)

bert_results_df.to_csv(TABLE_DIR / "bertimbau_repeated_cv_results_by_fold.csv", index=False)
bert_predictions_df.to_csv(PRED_DIR / "bertimbau_repeated_cv_predictions.csv", index=False)

if len(bert_results_df) > 0:
    bert_summary = (
        bert_results_df
        .groupby(["task", "model", "family"], dropna=False)
        .agg(
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_std=("macro_f1", "std"),
            weighted_f1_mean=("weighted_f1", "mean"),
            weighted_f1_std=("weighted_f1", "std"),
            accuracy_mean=("accuracy", "mean"),
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            qwk_mean=("quadratic_weighted_kappa", "mean"),
            qwk_std=("quadratic_weighted_kappa", "std"),
            mae_mean=("mae_ordinal", "mean"),
            mae_std=("mae_ordinal", "std"),
        )
        .reset_index()
    )
else:
    bert_summary = pd.DataFrame()

bert_summary.to_csv(TABLE_DIR / "bertimbau_repeated_cv_summary.csv", index=False)
display(bert_summary)

In [ ]:
# ============================================================
# 7. Unified model results and selected TF-IDF models
# ============================================================

tfidf_unified = tfidf_summary.copy() if "tfidf_summary" in globals() else pd.DataFrame()
if len(tfidf_unified) > 0:
    tfidf_unified["evaluation"] = "repeated_stratified_cv"
    tfidf_unified["n_seeds"] = len(SPLIT_SEEDS)

bert_unified = bert_summary.copy() if "bert_summary" in globals() else pd.DataFrame()
if len(bert_unified) > 0:
    bert_unified["input"] = "raw"
    bert_unified["evaluation"] = "repeated_stratified_cv"
    bert_unified["n_seeds"] = len(SPLIT_SEEDS)

all_summary = pd.concat([tfidf_unified, bert_unified], ignore_index=True, sort=False)
all_summary.to_csv(TABLE_DIR / "all_models_summary_final_1497.csv", index=False)

best_by_task = (
    all_summary.sort_values("macro_f1_mean", ascending=False)
    .groupby("task")
    .head(10)
    .sort_values(["task", "macro_f1_mean"], ascending=[True, False])
) if len(all_summary) > 0 else pd.DataFrame()

best_by_task.to_csv(TABLE_DIR / "best_models_by_task_final_1497.csv", index=False)

display(all_summary.sort_values(["task", "macro_f1_mean"], ascending=[True, False]))
display(best_by_task)

def best_tfidf_row(task):
    rows = tfidf_summary[tfidf_summary["task"] == task].sort_values("macro_f1_mean", ascending=False)
    if len(rows) == 0:
        raise ValueError(f"No TF-IDF results for {task}")
    return rows.iloc[0].to_dict()

def fit_final_tfidf(task):
    row = best_tfidf_row(task)
    data, X_raw, X_clean, y = get_task_dataset(task)
    X = X_clean if row["input"] == "stopfiltered" else X_raw
    pipe = make_tfidf_model(row["model"])
    pipe.fit(X, y)
    return {"pipeline": pipe, "input": row["input"], "selection": row}

final_tfidf_models = {}
for task in TASKS_TO_RUN:
    final_tfidf_models[task] = fit_final_tfidf(task)
    print("Final TF-IDF selected for", task, ":", final_tfidf_models[task]["selection"])

selected_tfidf_df = pd.DataFrame([{**{"task": task}, **final_tfidf_models[task]["selection"]} for task in final_tfidf_models])
selected_tfidf_df.to_csv(TABLE_DIR / "selected_tfidf_final_models.csv", index=False)

In [ ]:
# ============================================================
# 8. Final BERTimbau models for hybrid projection
# ============================================================

def train_final_bertimbau(task):
    import torch
    from torch import nn
    from datasets import Dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer,
        DataCollatorWithPadding,
        set_seed,
    )

    set_seed(RANDOM_STATE)

    data, X_raw, _, y = get_task_dataset(task)
    label_to_index, index_to_label = task_mapping(task)

    y_idx = np.array([label_to_index[int(x)] for x in y])
    num_labels = len(label_to_index)
    weights_np, counts = class_weights_from_labels(y_idx, num_labels)
    weights = torch.tensor(weights_np, dtype=torch.float)

    tokenizer = AutoTokenizer.from_pretrained(BERTIMBAU_MODEL_NAME)
    ds = Dataset.from_dict({"text": list(X_raw), "label": y_idx})

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

    ds = ds.map(tok, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(BERTIMBAU_MODEL_NAME, num_labels=num_labels)

    out_dir = RUNTIME_DIR / f"final_bert_{task}"
    out_dir.mkdir(parents=True, exist_ok=True)

    kwargs = dict(
        output_dir=str(out_dir),
        save_strategy="no",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        report_to=[],
        fp16=torch.cuda.is_available(),
        seed=RANDOM_STATE,
    )

    try:
        args = TrainingArguments(eval_strategy="no", **kwargs)
    except TypeError:
        args = TrainingArguments(evaluation_strategy="no", **kwargs)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.get("labels")
            outputs = model(**inputs)
            logits = outputs.get("logits")
            loss_fct = nn.CrossEntropyLoss(weight=weights.to(logits.device))
            loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
    )

    trainer.train()

    info = {
        "task": task,
        "train_n": len(data),
        "class_counts": json.dumps(counts.tolist()),
        "class_weights": json.dumps(weights_np.tolist()),
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }

    return {"trainer": trainer, "tokenizer": tokenizer, "index_to_label": index_to_label, "info": info}

def bert_predict(final_model_dict, texts):
    from datasets import Dataset
    import numpy as np

    trainer = final_model_dict["trainer"]
    tokenizer = final_model_dict["tokenizer"]
    index_to_label = final_model_dict["index_to_label"]

    ds = Dataset.from_dict({"text": list(texts)})

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

    ds = ds.map(tok, batched=True)
    out = trainer.predict(ds)
    pred_idx = np.argmax(out.predictions, axis=-1)
    return np.array([index_to_label[int(i)] for i in pred_idx])

# Previous decision: BERTimbau for applicability and intensity_3, TF-IDF for acceptance_3 and fine 5-class exploratory axes.
HYBRID_BERT_TASKS = ["applicability", "intensity_3"]

# If best_by_cv is selected, train BERT only for tasks where BERTimbau won and is available.
if HYBRID_POLICY == "best_by_cv" and len(best_by_task) > 0:
    tmp = []
    for task in ["applicability", "acceptance_3", "intensity_3"]:
        row = best_by_task[best_by_task["task"] == task].sort_values("macro_f1_mean", ascending=False).head(1)
        if len(row) and str(row.iloc[0]["family"]) == "transformer":
            tmp.append(task)
    HYBRID_BERT_TASKS = tmp

final_bert_models = {}
final_bert_info = []

if RUN_HYBRID_BEST_PROJECTION and RUN_FINAL_BERTIMBAU_FOR_HYBRID_PROJECTION and transformers_available():
    for task in HYBRID_BERT_TASKS:
        if task not in BERT_TASKS:
            print(f"Skipping final BERT for {task}: not in BERT_TASKS.")
            continue
        print("Training final BERTimbau for projection:", task)
        final_bert_models[task] = train_final_bertimbau(task)
        final_bert_info.append(final_bert_models[task]["info"])

pd.DataFrame(final_bert_info).to_csv(TABLE_DIR / "final_bertimbau_projection_training_info.csv", index=False)
display(pd.DataFrame(final_bert_info))

In [ ]:
# ============================================================
# 9. Projection utilities
# ============================================================

def prepare_projection_df(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    if "text" not in df.columns:
        for c in ["texto", "post_text", "content", "body"]:
            if c in df.columns:
                df["text"] = df[c]
                break

    if "text" not in df.columns:
        raise ValueError("Projection data needs a text-like field.")

    if "item_id" not in df.columns:
        if "index" in df.columns:
            df["item_id"] = pd.to_numeric(df["index"], errors="coerce")
            if df["item_id"].min() == 0:
                df["item_id"] = df["item_id"] + 1
        else:
            df["item_id"] = range(1, len(df) + 1)

    df["item_id"] = to_int_series(df["item_id"]).astype(int)
    df["text"] = df["text"].fillna("").astype(str)
    df["text_clean"] = df["text"].apply(clean_for_feature_analysis)
    df["text_no_stop"] = df["text"].apply(remove_pt_stopwords_text)
    return df

def predict_tfidf(task, df):
    item = final_tfidf_models[task]
    X = df["text_no_stop"].fillna("").astype(str) if item["input"] == "stopfiltered" else df["text"].fillna("").astype(str)
    return item["pipeline"].predict(X)

def predict_bert_or_fallback(task, df):
    if task in final_bert_models:
        return bert_predict(final_bert_models[task], df["text"].fillna("").astype(str).tolist())
    return predict_tfidf(task, df)

def add_linear_projection(df):
    out = df.copy()
    out["linear_applicability"] = predict_tfidf("applicability", out)

    mask = out["linear_applicability"] == 1
    for col in ["acceptance_3", "intensity_3", "acceptance_5", "intensity_5"]:
        out[f"linear_{col}"] = np.nan

    if mask.any():
        sub = out.loc[mask].copy()
        out.loc[mask, "linear_acceptance_3"] = predict_tfidf("acceptance_3", sub)
        out.loc[mask, "linear_intensity_3"] = predict_tfidf("intensity_3", sub)
        if "acceptance_5" in final_tfidf_models:
            out.loc[mask, "linear_acceptance_5"] = predict_tfidf("acceptance_5", sub)
        if "intensity_5" in final_tfidf_models:
            out.loc[mask, "linear_intensity_5"] = predict_tfidf("intensity_5", sub)

    return out

def add_hybrid_projection(df):
    out = df.copy()

    out["hybrid_applicability"] = predict_bert_or_fallback("applicability", out)

    mask = out["hybrid_applicability"] == 1
    for col in ["acceptance_3", "intensity_3", "acceptance_5", "intensity_5"]:
        out[f"hybrid_{col}"] = np.nan

    if mask.any():
        sub = out.loc[mask].copy()

        # Previous decision: TF-IDF for acceptance_3, BERTimbau for intensity_3 when available.
        out.loc[mask, "hybrid_acceptance_3"] = predict_tfidf("acceptance_3", sub)
        out.loc[mask, "hybrid_intensity_3"] = predict_bert_or_fallback("intensity_3", sub)

        # Fine 5x5 exploratory projection remains TF-IDF.
        if "acceptance_5" in final_tfidf_models:
            out.loc[mask, "hybrid_acceptance_5"] = predict_tfidf("acceptance_5", sub)
        if "intensity_5" in final_tfidf_models:
            out.loc[mask, "hybrid_intensity_5"] = predict_tfidf("intensity_5", sub)

    return out

def projection_matrix_3(df, prefix):
    app = df[df[f"{prefix}_applicability"] == 1].copy()
    if len(app) == 0:
        return pd.DataFrame(0, index=INTENSITY_ORDER_3, columns=ACCEPTANCE_ORDER_3)
    return pd.crosstab(app[f"{prefix}_intensity_3"], app[f"{prefix}_acceptance_3"]).reindex(index=INTENSITY_ORDER_3, columns=ACCEPTANCE_ORDER_3, fill_value=0)

def projection_matrix_5(df, prefix):
    app = df[df[f"{prefix}_applicability"] == 1].copy()
    if len(app) == 0:
        return pd.DataFrame(0, index=INTENSITY_ORDER_5, columns=ACCEPTANCE_ORDER_5)
    return pd.crosstab(app[f"{prefix}_intensity_5"], app[f"{prefix}_acceptance_5"]).reindex(index=INTENSITY_ORDER_5, columns=ACCEPTANCE_ORDER_5, fill_value=0)

def run_projection(name, df):
    df = prepare_projection_df(df)

    outputs = {}

    if RUN_LINEAR_ONLY_PROJECTION:
        linear = add_linear_projection(df)
        linear.to_csv(PRED_DIR / f"{name}_linear_predictions.csv", index=False)

        m3 = projection_matrix_3(linear, "linear")
        m5 = projection_matrix_5(linear, "linear")

        m3.to_csv(TABLE_DIR / f"{name}_linear_3x3_matrix.csv")
        m5.to_csv(TABLE_DIR / f"{name}_linear_5x5_matrix.csv")

        plot_bubble_matrix(m3, f"{name}: linear 3x3 projection", FIG_DIR / f"{name}_linear_3x3_matrix.png", kind="3x3")
        plot_bubble_matrix(m5, f"{name}: linear 5x5 projection", FIG_DIR / f"{name}_linear_5x5_matrix.png", kind="5x5")

        outputs["linear"] = linear

    if RUN_HYBRID_BEST_PROJECTION:
        hybrid = add_hybrid_projection(df)
        hybrid.to_csv(PRED_DIR / f"{name}_hybrid_predictions.csv", index=False)

        m3 = projection_matrix_3(hybrid, "hybrid")
        m5 = projection_matrix_5(hybrid, "hybrid")

        m3.to_csv(TABLE_DIR / f"{name}_hybrid_3x3_matrix.csv")
        m5.to_csv(TABLE_DIR / f"{name}_hybrid_5x5_matrix.csv")

        plot_bubble_matrix(m3, f"{name}: hybrid 3x3 projection", FIG_DIR / f"{name}_hybrid_3x3_matrix.png", kind="3x3")
        plot_bubble_matrix(m5, f"{name}: hybrid 5x5 projection", FIG_DIR / f"{name}_hybrid_5x5_matrix.png", kind="5x5")

        outputs["hybrid"] = hybrid

    return outputs

In [ ]:
# ============================================================
# 10. External 2024 projection
# ============================================================

external_path = maybe_locate_file(
    EXTERNAL_2024_FILE_NAME,
    [REPO_ROOT / "data" / "external_2024"],
)

external_projection_outputs = {}

if RUN_EXTERNAL_2024_PROJECTION and external_path is not None:
    print("External corpus located:", external_path)
    external_df = load_json_or_jsonl(external_path)
    external_projection_outputs = run_projection("external_2024", external_df)
elif RUN_EXTERNAL_2024_PROJECTION:
    print("External 2024 corpus not found. Skipping projection.")
else:
    print("External projection disabled.")